In [1]:
import tifffile
import os
import xml.etree.ElementTree as ET 

*Note* - Users should update paths in their own directory.

In [5]:
isilon_base = "/home/boyesh/IMLAkoyafusion"
db_path = '/home/boyesh/akoya_pcf/akoya_db' # Insert your database path here

def scan_for_files(base_path):
    """
    Scan base_path for QPTIFF files and returns a list of file paths
    """
    file_path = []

    # Ensure file exists

    if not os.path.exists(base_path):
        print('Folder not found!')
        return []

    # Find .qptiff files in the folder

    for root, dirs, files in os.walk(base_path):
        for file in files:
            if file.endswith(".qptiff"):
                full_path = os.path.join(root, file)
                file_path.append(full_path)
    return file_path

In [6]:
files = scan_for_files(isilon_base)

with tifffile.TiffFile(files[0]) as tif:
        t_series = tif.series[0]

        print(t_series.shape)
        print(t_series.axes)

(8, 37440, 30720)
CYX


In [7]:
with tifffile.TiffFile(files[0]) as tif:
    print(tif.pages[0].description)

<?xml version="1.0" encoding="utf-16"?>
<PerkinElmer-QPI-ImageDescription>
  <DescriptionVersion>6</DescriptionVersion>
  <AcquisitionSoftware>PhenoImagerHT 2.1.0</AcquisitionSoftware>
  <ImageType>FullResolution</ImageType>
  <Identifier>422cbd75-d801-4ee0-b8d0-230618d5a5ab</Identifier>
  <SlideID>052024 P7HuP120 #03 SG03</SlideID>
  <Barcode />
  <ComputerName>POLARIS</ComputerName>
  <IsUnmixedComponent>True</IsUnmixedComponent>
  <ExposureTime>2500</ExposureTime>
  <SignalUnits>40</SignalUnits>
  <ExposureTimeArray>
    <Value>2500</Value>
  </ExposureTimeArray>
  <Name>DAPI</Name>
  <Color>0,0,255</Color>
  <Responsivity>
    <Band>
      <Name>DAPI</Name>
      <Response>147.7713971048</Response>
      <Date>2024-06-03T14:59:14.1043813Z</Date>
      <FilterID>DAPI_Semrock:FF01-453/571/709-25 Emission / Semrock:FF01-391/538/649-25 Excitation</FilterID>
    </Band>
  </Responsivity>
  <Objective>10x</Objective>
  <Biomarker>DAPI</Biomarker>
  <NormalizedUnitsFactor>0.155544441</Nor

In [8]:
with tifffile.TiffFile(files[0]) as tif:
    meta = ET.fromstring(tif.pages[0].description)
    print(meta.find("SlideID").text)
    print(meta.find("Name").text)

052024 P7HuP120 #03 SG03
DAPI


In [9]:
with tifffile.TiffFile(files[0]) as tif:
    markers = []
    for page in tif.series[0].pages:
        meta = ET.fromstring(page.description)
        name = meta.find('Name')
        if name is not None:
            markers.append(name.text)
    print(markers)

['DAPI', 'Opal 480', 'Opal 520', 'Opal 570', 'Opal 780', 'Opal 620', 'Opal 690', 'Sample AF']


In [ ]:
def extract_metadata(file_path):
    """
    Extract metadata from a QPTIFF file. Returns a dict.
    """

    with tifffile.TiffFile(file_path) as tif:
        t_series = tif.series[0]
        meta = ET.fromstring(tif.pages[0].description)
        slide = meta.find("SlideID").text

        markers = []
        for page in tif.series[0].pages:
            meta = ET.fromstring(page.description)
            names = meta.find('Name')
            if names is not None:
                markers.append(names.text)
            
    meta_dict = {
        'file_path': file_path,
        'file_name': os.path.basename(file_path),
        'slide_id': slide,
        'num_channels': t_series.shape[0],
        'markers': markers,
        'image_shape': t_series.shape[1:]
    }

    return meta_dict

In [ ]:
res = extract_metadata(files[0])

res

In [ ]:
def validate_file(metadata):
    """
    Check metadata for completeness. Returns (bool, str) - valid message.
    """
    if not metadata['file_path']:
        return (False, "Missing file path")
    
    elif not metadata["file_name"]:
        return (False, "Missing file name")

    elif not metadata["slide_id"]:
        return (False, "Missing slide ID")
    
    elif not metadata['num_channels']:
        return (False, "Missing number of channels")
    
    elif not metadata['markers']:
        return (False, "Missing marker names")

    elif metadata['num_channels'] != len(metadata['markers']):
        return (False, "Number of channels does not match number of markers")

    elif not metadata['image_shape']:
        return (False, "Missing image dimensions")
    
    return (True, "All checks passed!")

In [ ]:
validate_file(res)

In [10]:
### Create DB for storage

import sqlite3

with open("schema", "r") as f:
   schema = f.read()

conn = sqlite3.connect("db_path")
conn.executescript(schema)
conn.commit
conn.close()

FileNotFoundError: [Errno 2] No such file or directory: 'schema'

In [ ]:
def write_to_db(db_path, metadata):
    """
    Write a validated file record to the Database.
    """

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("insert into slides (slide_name, file_path, file_name, num_channels) values (?, ?, ?, ?)",
                (metadata['slide_id'], metadata['file_path'], metadata['file_name'], metadata['num_channels']))
    slide_id = cursor.lastrowid
    cursor.execute("""INSERT INTO pipeline_status (slide_id, ingestion, preprocessing_qc, segmentation, feature_extraction, phenotyping, spatial_analysis) 
               VALUES (?, ?, ?, ?, ?, ?, ?)""", 
               (slide_id, 'Not Started', 'Not Started', 'Not Started', 'Not Started', 'Not Started', 'Not Started'))
    conn.commit()
    conn.close()

In [ ]:
write_to_db(db_path, res)

In [ ]:
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute("SELECT * FROM slides")
print(cursor.fetchall())
cursor.execute("SELECT * FROM pipeline_status")
print(cursor.fetchall())
conn.close()

In [ ]:
files = scan_for_files(db_path)
for file in files:
    metadata = extract_metadata(file)
    valid, message = validate_file(metadata)
    if valid:
        write_to_db(db_path, metadata)
        print(f"Written: {metadata['file_name']}")
    else:
        print(f"Skipped: {metadata['file_name']} — {message}")

In [ ]:
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute("SELECT * FROM slides")
print(cursor.fetchall())
cursor.execute("SELECT * FROM pipeline_status")
print(cursor.fetchall())
conn.close()


In [ ]:
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute("DELETE FROM slides")
cursor.execute("DELETE FROM pipeline_status")
conn.commit()
conn.close()

In [11]:
import sqlite3

with open("/home/boyesh/akoya_pcf/schema.sql", "r") as f:
    schema = f.read()

conn = sqlite3.connect("/home/boyesh/akoya_pcf/akoya.db")
conn.executescript("DROP TABLE IF EXISTS channel_stats; DROP TABLE IF EXISTS pipeline_status; DROP TABLE IF EXISTS slides; DROP TABLE IF EXISTS runs;")
conn.executescript(schema)
conn.commit()
conn.close()
print("Database recreated")

Database recreated
